In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"

import sys
sys.path.append('../../src/')

import numpy as onp
import jax
import jax.numpy as jnp

from myutils_corrected import Ntime, ACFs, set_detectors, set_detector_locations
from myutils_corrected import ln_likelihood_full_jit, Ntime, ACFs

from scipy.linalg import toeplitz, solve_triangular
import scipy.signal as sig

import lal
from gwpy.timeseries import TimeSeries
from other_utils import bandpass_ds, analysis_data, interp1d_jax, load_tables

jax.config.update("jax_enable_x64", True) 

/Users/kallol/Work/Misc/ringdown__2/examples/full-sky/../../src/myutils_corrected.py:3: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from chainconsumer import Chain, ChainConsumer, Truth, ChainConfig, PlotConfig

import numpyro
from numpyro.contrib.nested_sampling import NestedSampler
import numpyro.distributions as dist
numpyro.enable_x64()

/Users/kallol/miniconda3/envs/skyloc_ringdown/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
plt.rcParams["axes.grid"] = False

In [4]:
tgps = 1420878141.235932
tM = 0.337e-3 
tgps = tgps + 6*tM

seglen = 8
fs = 16384
fmin = 30
fmax       = 1500
event_id = "GW250114"
plot_checks = 0
T = 0.2
srate = 4096

ra = 2.33
dec = 0.190

factor = 10
seed       = 31567
t0         = 0.0

qnm1_path  = "../../data/l2/n1l2m2.dat"
qnm2_path  = "../../data/l2/n2l2m2.dat"

tM_shifted_samples_save_dir = "./GW250114-t-shift-posteriors/"

In [5]:
dH1 = TimeSeries.fetch_open_data('H1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)
dL1 = TimeSeries.fetch_open_data('L1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)

In [6]:
n_analyze = Ntime(srate, T)

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('H1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].location, ra, dec, tgps))
tH1 = tgps + dt_ifo

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('L1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].location, ra, dec, tgps))
tL1 = tgps + dt_ifo


dH1_cond = bandpass_ds(dH1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)
dL1_cond = bandpass_ds(dL1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)

nperseg = int(seglen * (1/(dH1.times.value[1] - dH1.times.value[0])))
noverlap = nperseg // 2

psdH = sig.welch(
    dH1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",   
)

psdL = sig.welch(
    dL1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density", 
)

In [7]:
dt_np = 1.0 / srate
Nt     = Ntime(srate=srate, T=T)
freqs_np = onp.fft.rfftfreq(Nt, d=dt_np)

fmin_eff = freqs_np[0]  if (fmin is None) else float(fmin)
fmax_eff = freqs_np[-1] if (fmax is None) else float(fmax)

i0 = int(onp.searchsorted(freqs_np, fmin_eff, side="left"))
i1 = int(onp.searchsorted(freqs_np, fmax_eff, side="right"))

freqs = jnp.asarray(freqs_np, dtype=jnp.float64)

# Prior limits
limits = [
    [30.0, 95.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0, 6.28],
    [-1., 1.],
    [0, 3.14]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

In [8]:
psdH = interp1d_jax(jnp.asarray(psdH[0],dtype=jnp.float64), jnp.asarray(psdH[1],dtype=jnp.float64))
psdL = interp1d_jax(jnp.asarray(psdL[0],dtype=jnp.float64), jnp.asarray(psdL[1],dtype=jnp.float64))

omega_r, omega_i, omega_OT_r, omega_OT_i = load_tables(qnm1_path, qnm2_path)
tgps = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps)

rhoH, rhoL = ACFs(srate=srate, T=T, psdH=psdH, psdL=psdL, factor=factor)
covH=toeplitz(rhoH)
covL=toeplitz(rhoL)
L_H=jnp.linalg.cholesky(covH)
L_L=jnp.linalg.cholesky(covL)

resp_H_py = lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].response
resp_L_py = lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].response
resp_H = jnp.asarray(resp_H_py, dtype=jnp.float64)
resp_L = jnp.asarray(resp_L_py, dtype=jnp.float64)

set_detectors(resp_H, resp_L)

H1 = lal.CachedDetectors[lal.LALDetectorIndexLHODIFF]
L1 = lal.CachedDetectors[lal.LALDetectorIndexLLODIFF]

rH = jnp.array([H1.location[0], H1.location[1], H1.location[2]], dtype=jnp.float64)
rL = jnp.array([L1.location[0], L1.location[1], L1.location[2]], dtype=jnp.float64)

set_detector_locations(rH, rL)

In [9]:
num_shifts = 15

t_shifts = []
H1_data_t_shifted = []
L1_data_t_shifted = []

for jj in range(num_shifts):
    dH1_analysis_data = analysis_data(dH1_cond[1], dH1_cond[0], tH1 + jj*1*tM, n_analyze)
    dL1_analysis_data = analysis_data(dL1_cond[1], dL1_cond[0], tH1 + jj*1*tM, n_analyze)

    t_shifts.append(jj*1*tM)
    H1_data_t_shifted.append(dH1_analysis_data[1])
    L1_data_t_shifted.append(dL1_analysis_data[1])

In [10]:
onp.savetxt(tM_shifted_samples_save_dir + 't_shifts.txt', t_shifts)

In [11]:
low  = jnp.asarray(low,  dtype=jnp.float64)
high = jnp.asarray(high, dtype=jnp.float64)

In [12]:
logZs = []
for i in range(num_shifts):
    h_H_t = H1_data_t_shifted[i]
    h_L_t = L1_data_t_shifted[i]

    def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                    omega_r, omega_i, omega_OT_r, omega_OT_i,
                    T, srate, t0):
        def _loglik(theta):
            return ln_likelihood_full_jit(
                dataH=dataH, dataL=dataL, params=theta,
                gmst=gmst, L_H=L_H, L_L=L_L,
                omega_r=omega_r, omega_i=omega_i,
                omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
                T=T, srate=srate, t0=t0
            )
        return _loglik

    loglik_fn = make_loglik_fn(
        dataH=h_H_t, dataL=h_L_t,
        gmst=gmst, L_H=L_H, L_L=L_L,
        omega_r=omega_r, omega_i=omega_i,
        omega_OT_r=omega_OT_r, omega_OT_i=omega_OT_i,
        T=T, srate=srate, t0=t0
    )

    def model():
        theta = numpyro.sample("theta", dist.Uniform(low, high).to_event(1))
        numpyro.factor("loglike", loglik_fn(theta))

    rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
    rng_run, rng_post = jax.random.split(rng_key)
    ns = NestedSampler(
        model,
        constructor_kwargs=dict(
            num_live_points=10000,   
            max_samples=500_000,    
            verbose=True,
        ),
        termination_kwargs=dict(
            dlogZ=0.01,            
        ),
    )
    ns.run(rng_run)
    ns.print_summary()
    posterior = ns.get_samples(rng_post, num_samples=100_000)

    onp.save(tM_shifted_samples_save_dir + 'posterior.' + event_id + '.' + str(i) + '.npy', onp.asarray(posterior['theta']))
    logZs.append(ns._results.log_Z_mean)


INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 298795
Efficiency: 0.03294283391321236
log(L) contour: -4238.569160599576
log(Z) est.: -776.6380493241124 +- 0.830887458548236
log(Z | remaining) est.: 3471.458794814922 +- 0.9952232666241158
ESS: 0.5027595396141746

-------
Num samples: 10008
Num likelihood evals: 646380
Efficiency: 0.02015482644455006
log(L) contour: -1424.5683584203027
log(Z) est.: -761.6168059044347 +- 0.685559361823379
log(Z | remaining) est.: 671.8015377101373 +- 0.7611274089269214
ESS: 0.8333599730089527

----

INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 297866
Efficiency: 0.033043880212632486
log(L) contour: -4164.665963091613
log(Z) est.: -756.3928205293503 +- 0.8312192104834675
log(Z | remaining) est.: 3419.009674101747 +- 1.1731569145376075
ESS: 0.5022040343013453

-------
Num samples: 10008
Num likelihood evals: 646523
Efficiency: 0.020053098231728697
log(L) contour: -1387.1122745083203
log(Z) est.: -756.8928193723169 +- 0.8312487944174309
log(Z | remaining) est.: 638.6028703556955 +- 0.8776629116981568
ESS: 0.5021797225437841



INFO:jaxns:Number of Markov-chains set to: 10000


...

--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 40887718
samples: 190152
phantom samples: 0
likelihood evals / sample: 215.0
phantom fraction (%): 0.0%
--------
logZ=-731.111 +- 0.043
max(logL)=-713.504
H=-12.78
ESS=27705
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 59.9 +- 9.4 | 48.9 / 58.5 / 72.5 | 48.9 | 48.9
theta[1]: 0.47 +- 0.24 | 0.12 / 0.5 / 0.78 | 0.21 | 0.21
theta[2]: 1.42 +- 0.95 | 0.41 / 1.12 / 2.91 | 2.25 | 2.25
theta[3]: 3.3 +- 1.9 | 0.7 / 3.0 / 5.8 | 1.2 | 1.2
theta[4]: 1.3 +- 1.0 | 0.2 / 1.0 / 3.0 | 4.0 | 4.0
theta[5]: 3.1 +- 1.7 | 0.8 / 3.1 / 5.4 | 3.7 | 3.7
theta[6]: -0.01 +- 0.41 | -0.54 / -0.03 / 0.56 | -0.22 | -0.22
theta[7]: 3.2 +- 1.2 | 1.7 / 3.3 / 4.8 | 1.2 | 1.2
theta[8]: -0.06 +- 0.53 | -0.81 / -0.08 / 0.65 | -0.91 | -0.91
theta[9]: 1.56 +- 0.91 | 0.29 / 1.64 / 2.82 | 2.58 | 2.58
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running unif

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 37374218
samples: 180144
phantom samples: 0
likelihood evals / sample: 207.5
phantom fraction (%): 0.0%
--------
logZ=-728.774 +- 0.042
max(logL)=-712.505
H=-12.32
ESS=25328
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 60.7 +- 9.7 | 49.7 / 59.1 / 73.9 | 52.1 | 52.1
theta[1]: 0.48 +- 0.25 | 0.12 / 0.5 / 0.8 | 0.26 | 0.26
theta[2]: 1.5 +- 1.0 | 0.4 / 1.2 / 3.1 | 3.6 | 3.6
theta[3]: 2.9 +- 1.8 | 0.5 / 2.9 / 5.7 | 5.8 | 5.8
theta[4]: 1.04 +- 0.92 | 0.14 / 0.72 / 2.52 | 3.18 | 3.18
theta[5]: 3.4 +- 1.8 | 0.8 / 3.5 / 5.8 | 2.3 | 2.3
theta[6]: -0.03 +- 0.41 | -0.57 / -0.03 / 0.54 | -0.07 | -0.07
theta[7]: 3.2 +- 1.2 | 1.7 / 3.4 / 4.8 | 3.9 | 3.9
theta[8]: -0.09 +- 0.54 | -0.83 / -0.09 / 0.67 | 0.29 | 0.29
theta[9]: 1.33 +- 0.96 | 0.15 / 1.22 / 2.79 | 0.13 | 0.13
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 35355320
samples: 170136
phantom samples: 0
likelihood evals / sample: 207.8
phantom fraction (%): 0.0%
--------
logZ=-728.271 +- 0.04
max(logL)=-713.25
H=-11.1
ESS=25569
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 62.0 +- 10.0 | 50.0 / 61.0 / 76.0 | 61.0 | 61.0
theta[1]: 0.5 +- 0.25 | 0.12 / 0.54 / 0.81 | 0.61 | 0.61
theta[2]: 1.4 +- 1.0 | 0.3 / 1.2 / 3.1 | 3.1 | 3.1
theta[3]: 3.1 +- 1.8 | 0.6 / 3.2 / 5.6 | 4.6 | 4.6
theta[4]: 0.96 +- 0.86 | 0.11 / 0.68 / 2.24 | 2.76 | 2.76
theta[5]: 3.0 +- 1.8 | 0.6 / 3.0 / 5.5 | 0.0 | 0.0
theta[6]: -0.04 +- 0.39 | -0.59 / -0.04 / 0.47 | -0.58 | -0.58
theta[7]: 3.3 +- 1.2 | 1.7 / 3.5 / 4.8 | 3.5 | 3.5
theta[8]: -0.18 +- 0.5 | -0.84 / -0.18 / 0.54 | -0.15 | -0.15
theta[9]: 1.51 +- 0.92 | 0.25 / 1.55 / 2.86 | 1.45 | 1.45
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform

In [13]:
logZs

[Array(-738.52850492, dtype=float64),
 Array(-738.23539068, dtype=float64),
 Array(-736.69817894, dtype=float64),
 Array(-734.65994481, dtype=float64),
 Array(-733.38216374, dtype=float64),
 Array(-733.1146658, dtype=float64),
 Array(-734.69933305, dtype=float64),
 Array(-734.07369833, dtype=float64),
 Array(-733.3748709, dtype=float64),
 Array(-732.76713241, dtype=float64),
 Array(-732.49365302, dtype=float64),
 Array(-731.14841888, dtype=float64),
 Array(-728.7703983, dtype=float64),
 Array(-728.00158905, dtype=float64),
 Array(-729.36602907, dtype=float64)]